# Module 03 - MLOps pipeline and the evaluation gate

Submit the `prep -> train -> evaluate gate` pipeline to the cluster and watch
promotion happen on merit. See [README.md](README.md).

In [ ]:
# Tutorial setup: find the repo root and make the repo code importable.
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "requirements.txt").exists() and (p / "src").exists():
            return p
    return start


REPO = find_repo_root(Path.cwd())
for sub in ["src", "data", "webapp/backend"]:
    sys.path.insert(0, str(REPO / sub))
print("Repo root:", REPO)

## Submit the pipeline

This runs on the `cpu-cluster`. We submit without streaming so the notebook stays
responsive, then open the job in Studio.

In [ ]:
import subprocess

proc = subprocess.run(
    [sys.executable, str(REPO / "src" / "pipeline" / "pipeline.py"),
     "--rmse-threshold", "80", "--no-stream"],
    cwd=str(REPO), capture_output=True, text=True,
)
print(proc.stdout[-2000:])
print(proc.stderr[-1000:])

## The gate logic

Open [src/pipeline/evaluate.py](../../src/pipeline/evaluate.py): the model is
registered only if RMSE beats the threshold, otherwise the step raises and the
pipeline stops.

## List registered model versions and their lineage

In [ ]:
from common.workspace import get_ml_client, load_resources

ml_client = get_ml_client()
name = load_resources().get("model_name", "aeso-price-forecaster")
for m in ml_client.models.list(name=name):
    print(f"{m.name}:{m.version}  tags={m.tags}")

## Next

Continue to [Module 04](../04-deploy-endpoint/README.md).